# 04. Model Dự đoán Độ lớn (XGBoost)

**Mục tiêu:** Train XGBoost model để dự đoán độ lớn động đất tiếp theo

**Input:** train_enhanced.csv
**Output:** Trained model, evaluation metrics

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
from pathlib import Path
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("="*60)
print(" XGBOOST - MAGNITUDE PREDICTION MODEL ")
print("="*60)

## 1. Load Dữ liệu

In [ ]:
# Load data
data_dir = Path('/home/haind/Desktop/earthquake-sequence-mining/haind/data')

train = pd.read_csv(data_dir / 'train_enhanced.csv')
val = pd.read_csv(data_dir / 'val_enhanced.csv')
test = pd.read_csv(data_dir / 'test_enhanced.csv')

# Load config
with open(data_dir / 'features.json', 'r') as f:
    config = json.load(f)

FEATURES = config['NUMERICAL_FEATURES']
TARGET_MAG = config['TARGET_MAG']

print(f"Train: {len(train):,} events")
print(f"Val: {len(val):,} events")
print(f"Test: {len(test):,} events")
print(f"\nFeatures: {len(FEATURES)}")

## 2. Chuẩn bị Datasets

In [ ]:
# Chuẩn bị features và target
X_train = train[FEATURES]
y_train = train[TARGET_MAG]

X_val = val[FEATURES]
y_val = val[TARGET_MAG]

X_test = test[FEATURES]
y_test = test[TARGET_MAG]

# Fill remaining NaN
X_train = X_train.fillna(0)
X_val = X_val.fillna(0)
X_test = X_test.fillna(0)

y_train = y_train.fillna(y_train.median())
y_val = y_val.fillna(y_val.median())
y_test = y_test.fillna(y_test.median())

print(f"\nX_train shape: {X_train.shape}")
print(f"y_train range: {y_train.min():.2f} đến {y_train.max():.2f}")
print(f"\nTarget statistics (train):")
print(y_train.describe())

## 3. Tạo DMatrix

In [ ]:
# Tạo DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

print("✓ Đã tạo DMatrix!")

## 4. Thiết lập Parameters

In [ ]:
# XGBoost parameters cho magnitude prediction
params = {
    'objective': 'reg:squarederror',
    'eval_metric': ['rmse', 'mae'],
    'max_depth': 6,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'random_state': 42,
    'tree_method': 'hist',
    'n_jobs': -1
}

print("Parameters:")
for key, value in params.items():
    print(f"  {key}: {value}")

## 5. Training Model

In [ ]:
# Training
evals_result = {}

print("\n" + "="*60)
print(" TRAINING MAGNITUDE MODEL ")
print("="*60)

model = xgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=100,
    evals_result=evals_result,
    verbose_eval=100
)

print(f"\n✓ Training hoàn thành!")
print(f"Best iteration: {model.best_iteration}")
print(f"Best score: {model.best_score:.4f}")

## 6. Training Curve

In [ ]:
# Vẽ training curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSE
axes[0].plot(evals_result['train']['rmse'], label='Train')
axes[0].plot(evals_result['val']['rmse'], label='Validation')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('RMSE (magnitude)')
axes[0].set_title('Training Curve - RMSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(evals_result['train']['mae'], label='Train')
axes[1].plot(evals_result['val']['mae'], label='Validation')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('MAE (magnitude)')
axes[1].set_title('Training Curve - MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluation

In [ ]:
# Predict
y_pred_val = model.predict(dval)
y_pred_test = model.predict(dtest)

# Metrics
r2_val = r2_score(y_val, y_pred_val)
mae_val = mean_absolute_error(y_val, y_pred_val)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))

r2_test = r2_score(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print("="*60)
print(" VALIDATION RESULTS ")
print("="*60)
print(f"R²:   {r2_val:.4f}")
print(f"MAE:  {mae_val:.3f} magnitude")
print(f"RMSE: {rmse_val:.3f} magnitude")

print("\n" + "="*60)
print(" TEST RESULTS ")
print("="*60)
print(f"R²:   {r2_test:.4f}")
print(f"MAE:  {mae_test:.3f} magnitude")
print(f"RMSE: {rmse_test:.3f} magnitude")

## 8. Feature Importance

In [ ]:
# Feature importance
importance = model.get_score(importance_type='weight')
importance_df = pd.DataFrame({
    'feature': importance.keys(),
    'importance': importance.values()
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=importance_df.head(15), x='importance', y='feature')
plt.title('Top 15 Feature Importance - Magnitude Prediction')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 9. Actual vs Predicted

In [ ]:
# Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_val, y_pred_val, alpha=0.3, s=10)
axes[0].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
axes[0].set_xlabel('Độ lớn thực')
axes[0].set_ylabel('Độ lớn dự đoán')
axes[0].set_title(f'Validation: R² = {r2_val:.4f}, MAE = {mae_val:.3f}')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_test, y_pred_test, alpha=0.3, s=10)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Độ lớn thực')
axes[1].set_ylabel('Độ lớn dự đoán')
axes[1].set_title(f'Test: R² = {r2_test:.4f}, MAE = {mae_test:.3f}')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Error Analysis theo Magnitude Range

In [ ]:
# Error theo magnitude bins
test['mag_bin'] = pd.cut(test['mag'], bins=[3, 4, 5, 6, 7, 10],
                          labels=['3-4', '4-5', '5-6', '6-7', '7+'])

mae_by_bin = test.groupby('mag_bin', observed=True).apply(
    lambda x: mean_absolute_error(x[TARGET_MAG],
                                   model.predict(xgb.DMatrix(x[FEATURES])))
)

print("\nMAE theo magnitude range:")
print(mae_by_bin)

# Plot
plt.figure(figsize=(10, 5))
mae_by_bin.plot(kind='bar', color='steelblue')
plt.xlabel('Magnitude Range')
plt.ylabel('MAE')
plt.title('Error theo Magnitude Range')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Lưu Model

In [ ]:
# Tạo thư mục model
model_dir = Path('/home/haind/Desktop/earthquake-sequence-mining/haind/models')
model_dir.mkdir(exist_ok=True)

# Lưu model
joblib.dump(model, model_dir / 'xgboost_mag_model.pkl')

# Lưu results
results = {
    'model': 'xgboost_magnitude',
    'best_iteration': int(model.best_iteration),
    'best_score': float(model.best_score),
    'test_r2': float(r2_test),
    'test_mae': float(mae_test),
    'test_rmse': float(rmse_test),
    'val_r2': float(r2_val),
    'val_mae': float(mae_val),
    'val_rmse': float(rmse_val),
    'features': FEATURES,
    'mae_by_bin': {str(k): float(v) for k, v in mae_by_bin.items()}
}

with open(model_dir / 'mag_model_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("="*60)
print(" ĐÃ LƯU MODEL! ")
print("="*60)
print(f"Model: {model_dir / 'xgboost_mag_model.pkl'}")
print(f"Results: {model_dir / 'mag_model_results.json'}")

## 12. Summary

In [ ]:
print("="*60)
print(" SUMMARY - MAGNITUDE MODEL ")
print("="*60)
print(f"\nValidation Results:")
print(f"  R²:   {r2_val:.4f}")
print(f"  MAE:  {mae_val:.3f} magnitude")
print(f"\nTest Results:")
print(f"  R²:   {r2_test:.4f}")
print(f"  MAE:  {mae_test:.3f} magnitude")
print(f"\nTop 5 Features:")
for i, row in importance_df.head(5).iterrows():
    print(f"  - {row['feature']}: {row['importance']:.0f}")
print(f"\nNext step: Run 05_evaluation.ipynb")